# Forest Fire Frequency and Impact Across Regions and Seasons

**Course:** Exploratory Data Analysis and Visualization (EDAV)  
**Dataset:** `forest_fire_dataset_one_sheet.xlsx`

This notebook follows the five project questions (Q1–Q5) and uses concepts from the syllabus: NumPy arrays, Pandas DataFrames, filtering, missing-value handling, grouping, date/time preparation, Matplotlib figures/subplots, and Seaborn plots.

> **Dataset note:** The provided file labels its records as *SIMULATED / EDUCATIONAL — not observed fire records*. The analysis is for learning and demonstration, not for reporting real-world fire statistics.


## 1. Import the required libraries
We use Pandas to work with the table, NumPy for numerical operations, Matplotlib to create figures and subplots, and Seaborn for clear statistical plots.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from google.colab import files

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)


## 2. Upload and load the Excel dataset
Run the cell and select `forest_fire_dataset_one_sheet.xlsx` from your computer.

In [ ]:
uploaded = files.upload()

# Read the uploaded Excel file
file_name = next(iter(uploaded))
df = pd.read_excel(file_name)

print("Dataset loaded successfully.")
print("Rows and columns:", df.shape)
display(df.head())


## 3. Understand the data
Check column names, data types, summary statistics, and missing values before doing the analysis.

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types and non-null values:")
df.info()

print("\nSummary of numeric columns:")
display(df.describe())

print("\nMissing values in each column:")
display(df.isna().sum())


## 4. Prepare the data
Convert the numeric columns safely, standardize the month order, and create a date column for monthly trend plots. The original rows are kept; missing numeric values are handled in a separate cleaned DataFrame below.

In [ ]:
# Convert numeric columns in case Excel stores any values as text
numeric_cols = ["Year", "Month_Number", "Fire_Frequency_Count",
                "Estimated_Burned_Area_km2"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Create a date column from Year and Month_Number
df["Date"] = pd.to_datetime(
    dict(year=df["Year"], month=df["Month_Number"], day=1),
    errors="coerce"
)

month_order = [
    "January", "February", "March", "April", "May", "June",
    "July", "August", "September", "October", "November", "December"
]
df["Month"] = pd.Categorical(df["Month"], categories=month_order, ordered=True)

print("Prepared data preview:")
display(df[["Region", "Year", "Month", "Hemisphere_Season",
            "Fire_Frequency_Count", "Estimated_Burned_Area_km2", "Date"]].head())


## Q1 (BL-3): Calculate average fire incidence

**Logic:** Select the `Fire_Frequency_Count` column and calculate its mean. Pandas skips missing values by default, so the result is the average of available fire counts.

In [ ]:
average_fire_incidence = df["Fire_Frequency_Count"].mean()
print(f"Average fire incidence (available records): {average_fire_incidence:.2f}")

# Optional: average burned area for impact context
average_burned_area = df["Estimated_Burned_Area_km2"].mean()
print(f"Average estimated burned area: {average_burned_area:.2f} km²")


## Q2 (BL-3): Filter fire data by region

**Logic:** Use Boolean indexing to keep only rows where `Region` matches the selected region. Change the region name if you want to examine a different place.

In [ ]:
selected_region = "California, USA"

region_data = df[df["Region"] == selected_region].copy()

print("Selected region:", selected_region)
print("Number of records:", len(region_data))
display(region_data[["Region", "Year", "Month", "Hemisphere_Season",
                     "Fire_Frequency_Count", "Estimated_Burned_Area_km2"]])


## Q3 (BL-4): Handle missing fire records

**Logic:** First count missing values. Then create a cleaned copy and fill missing numeric values with each column's median. Median is used because it is less affected by unusually large values than the mean. The original `df` is not overwritten.

In [ ]:
numeric_to_clean = ["Fire_Frequency_Count", "Estimated_Burned_Area_km2"]

print("Missing values before cleaning:")
display(df[numeric_to_clean].isna().sum())

clean_df = df.copy()
for col in numeric_to_clean:
    median_value = clean_df[col].median()
    clean_df[col] = clean_df[col].fillna(median_value)

print("\nMissing values after median filling:")
display(clean_df[numeric_to_clean].isna().sum())

print("Rows in original and cleaned data:", len(df), len(clean_df))


## Q4 (BL-4): Group fire incidences by season and region

**Logic:** Use `groupby()` to calculate the mean fire count for each season and for each region. Use `reset_index()` to make the grouped result a normal table.

In [ ]:
season_summary = (
    clean_df.groupby("Hemisphere_Season", observed=False)["Fire_Frequency_Count"]
    .mean()
    .reset_index(name="Average_Fire_Frequency")
    .sort_values("Average_Fire_Frequency", ascending=False)
)

region_summary = (
    clean_df.groupby("Region", observed=False)["Fire_Frequency_Count"]
    .mean()
    .reset_index(name="Average_Fire_Frequency")
    .sort_values("Average_Fire_Frequency", ascending=False)
)

print("Average fire frequency by season:")
display(season_summary)

print("Top 10 regions by average fire frequency:")
display(region_summary.head(10))


## Q5 (BL-5): Visualize fire frequency trends using subplots

The first figure contains two subplots: monthly fire frequency and estimated burned area. The second figure compares average fire frequency by season and the top 10 regions. These plots use Matplotlib's `subplots()` and Seaborn's line/bar plots.

In [ ]:
# Monthly summaries across all regions
monthly_summary = (
    clean_df.groupby(["Date"], as_index=False)
    .agg(
        Total_Fire_Frequency=("Fire_Frequency_Count", "sum"),
        Total_Burned_Area=("Estimated_Burned_Area_km2", "sum")
    )
    .sort_values("Date")
)

fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)

sns.lineplot(data=monthly_summary, x="Date", y="Total_Fire_Frequency",
             marker="o", ax=axes[0])
axes[0].set_title("Monthly Fire Frequency Across All Regions")
axes[0].set_xlabel("")
axes[0].set_ylabel("Total fire count")

sns.lineplot(data=monthly_summary, x="Date", y="Total_Burned_Area",
             marker="o", color="darkorange", ax=axes[1])
axes[1].set_title("Monthly Estimated Burned Area Across All Regions")
axes[1].set_xlabel("Month")
axes[1].set_ylabel("Estimated burned area (km²)")

plt.tight_layout()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.barplot(data=season_summary, x="Hemisphere_Season",
            y="Average_Fire_Frequency", color="steelblue", ax=axes[0])
axes[0].set_title("Average Fire Frequency by Season")
axes[0].set_xlabel("Season")
axes[0].set_ylabel("Average fire count")
axes[0].tick_params(axis="x", rotation=45)

top10 = region_summary.head(10).sort_values("Average_Fire_Frequency")
sns.barplot(data=top10, x="Average_Fire_Frequency", y="Region", color="darkorange", ax=axes[1])
axes[1].set_title("Top 10 Regions by Average Fire Frequency")
axes[1].set_xlabel("Average fire count")
axes[1].set_ylabel("Region")

plt.tight_layout()
plt.show()


## 5. Short conclusion
The tables and plots show how the dataset's fire counts differ across months, seasons, and regions. Missing numeric values are filled with medians in a separate cleaned DataFrame. Since the file is marked simulated/educational, the results should be described as patterns in the provided dataset rather than actual recorded forest-fire events.

In [ ]:
print("PROJECT SUMMARY")
print("----------------")
print(f"Total records: {len(df)}")
print(f"Number of regions: {df['Region'].nunique()}")
print(f"Average available fire count: {df['Fire_Frequency_Count'].mean():.2f}")
print(f"Missing fire counts originally: {df['Fire_Frequency_Count'].isna().sum()}")
print("Dataset type:", df["Data_Type"].dropna().iloc[0] if "Data_Type" in df and df["Data_Type"].notna().any() else "Not specified")
